# Music Annotation Project 60R2
## Verified 59P observed-print inference → verified 59R1 REMOVE inference

**Runtime:** GPU recommended. Select a T4 GPU, then **Runtime → Run all**.

This notebook is rebuilt as a self-contained integration notebook. It:

1. resolves the authoritative Notebook 07 reviewed pages and masks;
2. verifies and loads the frozen 59P observed-print bundle;
3. generates page-shaped calibrated `p_print` directly from the 59P model;
4. verifies and loads the frozen 59R1 REMOVE bundle;
5. constructs all 17 frozen REMOVE predictors, including `boundary_distance` and `distance_to_protected`;
6. applies the frozen REMOVE model, calibrator, threshold, and page-global coherence rule;
7. forms `M5_REMOVE = M3 & ~SAFE_REMOVE`;
8. reconstructs from `M5_REMOVE` and enforces outside-mask immutability;
9. writes page artefacts, probabilities, decisions, hashes, manifests, and a curated upload ZIP.

### Safety boundary

- Notebook 07's corrected mask is the sole intervention authority.
- Notebook 06 YOLO confidence is not used as `p_print`.
- `p_print` is generated only by the verified 59P observed-print model and calibrator.
- SAFE_PROTECT is disabled.
- Missing, mismatched, ambiguous, or non-finite inputs fail closed.
- This notebook has no production-promotion authority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow matplotlib

## 1. Imports, configuration, paths, and common helpers

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, random, gc, time, zipfile

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Set these to exact run roots if automatic discovery is not appropriate.
OBSERVED_PRINT_RUN_OVERRIDE = None   # 59P run root, not bundle/
REMOVE_RUN_OVERRIDE = None           # 59R1 run root, not bundle/

# Notebook 07 audit directory.
REVIEW_AUDIT_DIR = ROOT / 'human_in_the_middle' / 'audit'

# Candidate roots used only for strict complete-run discovery.
OBSERVED_PRINT_ROOTS = [
    ROOT / 'observed_print_deployment_freeze_v8_2',
    ROOT / 'observed_print_deployment_fit_and_freeze_v8_2',
    ROOT / 'observed_print_classifier_v8_2_deployment_freeze',
]
REMOVE_ROOTS = [
    ROOT / 'observable_residual_mask_m5_v8_7_deployment_freeze',
    ROOT / 'remove_deployment_freeze_v8_8',
    ROOT / 'remove_deployment_fit_and_freeze_v8_8',
]

OUTPUT_ROOT = ROOT / 'notebook60r2_verified_integrated_runtime'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook60r2'
OUT = OUTPUT_ROOT / RUN_ID
PAGES_OUT = OUT / 'pages'
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for p in [OUT, PAGES_OUT, TABLES, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

SEED = 42
BATCH_SIZE = 2048
EXPECTED_REVIEWED_PAGES = 151
REQUIRE_EXACT_151 = True
PAGE_ALLOWLIST = None

# Frozen V7.8 constants carried into both deployment bundles.
MASK_DILATION = 2
CROP_MARGIN = 48
GEOMETRY_THRESHOLD = 80
BAND_WIDTH = 4
DARK_THRESHOLD = 200
EDGE_LOW = 70
EDGE_HIGH = 170
LOCAL_RADIUS = 3

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SUPPORTED_IMAGES = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}


def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def read_json(path):
    path = Path(path)
    require(path.exists(), f'Missing JSON file: {path}')
    value = json.loads(path.read_text())
    require(isinstance(value, dict), f'JSON root is not an object: {path}')
    return value


def atomic_json(path, value):
    path = Path(path)
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def array_sha256(array):
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())
    return h.hexdigest()


def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Unreadable greyscale image: {path}')
    return image


def binary_mask(mask):
    return np.where(np.asarray(mask) > 0, 255, 0).astype(np.uint8)


def immutable(original, candidate, mask):
    output = original.copy()
    output[mask > 0] = candidate[mask > 0]
    require(
        np.array_equal(output[mask == 0], original[mask == 0]),
        'Candidate changed a pixel outside the authoritative mask.',
    )
    return output


def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, separators=(',', ':')).encode()
    ).hexdigest()

print('Output:', OUT)
print('Device:', DEVICE)

## 2. Resolve, verify, and load the 59P observed-print bundle

In [ ]:
OP_BUNDLE_FILES = [
    'observed_print_model.pt',
    'observed_print_calibrator.joblib',
    'channel_schema.json',
    'policy.json',
    'runtime_contract.json',
    'preprocessing_identity.json',
    'software_environment.json',
    'freeze_manifest.json',
]
OP_REPLAY_RELATIVE = Path('replay') / 'replay_test.json'


def normalise_run_override(value):
    if value is None:
        return None
    path = Path(value)
    return path.parent if path.name == 'bundle' else path


def complete_run(run, bundle_files, replay_relative):
    bundle = run / 'bundle'
    return (
        run.is_dir()
        and bundle.is_dir()
        and all((bundle / name).is_file() for name in bundle_files)
        and (run / replay_relative).is_file()
    )


def discover_complete_run(roots, override, bundle_files, replay_relative, label):
    override = normalise_run_override(override)
    if override is not None:
        require(
            complete_run(override, bundle_files, replay_relative),
            f'{label} override is incomplete: {override}',
        )
        return override

    hits = []
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.iterdir():
            if complete_run(candidate, bundle_files, replay_relative):
                hits.append(candidate)
    require(hits, f'No complete {label} run found. Set its run override explicitly.')
    return max(hits, key=lambda p: p.stat().st_mtime)


OP_RUN = discover_complete_run(
    OBSERVED_PRINT_ROOTS,
    OBSERVED_PRINT_RUN_OVERRIDE,
    OP_BUNDLE_FILES,
    OP_REPLAY_RELATIVE,
    '59P observed-print deployment',
)
OP_BUNDLE = OP_RUN / 'bundle'
OP_REPLAY_PATH = OP_RUN / OP_REPLAY_RELATIVE

OP_MODEL_PATH = OP_BUNDLE / 'observed_print_model.pt'
OP_CALIBRATOR_PATH = OP_BUNDLE / 'observed_print_calibrator.joblib'
OP_SCHEMA_PATH = OP_BUNDLE / 'channel_schema.json'
OP_POLICY_PATH = OP_BUNDLE / 'policy.json'
OP_CONTRACT_PATH = OP_BUNDLE / 'runtime_contract.json'
OP_PREPROCESSING_PATH = OP_BUNDLE / 'preprocessing_identity.json'
OP_ENVIRONMENT_PATH = OP_BUNDLE / 'software_environment.json'
OP_FREEZE_PATH = OP_BUNDLE / 'freeze_manifest.json'

op_schema = read_json(OP_SCHEMA_PATH)
op_policy = read_json(OP_POLICY_PATH)
op_contract = read_json(OP_CONTRACT_PATH)
op_preprocessing = read_json(OP_PREPROCESSING_PATH)
op_freeze = read_json(OP_FREEZE_PATH)
op_replay = read_json(OP_REPLAY_PATH)

op_checks = {
    'deployment_bundle_complete': op_freeze.get('deployment_bundle_complete') is True,
    'replay_passed': op_replay.get('passed') is True,
    'model_reload_exact': op_replay.get('model_reload_exact') is True,
    'calibrator_reload_exact': op_replay.get('calibrator_reload_exact') is True,
    'decision_reload_exact': op_replay.get('decision_reload_exact') is True,
}

# Verify manifest objects. Relative paths are relative to bundle/ unless explicitly ../replay/.
for item in op_freeze.get('objects', []):
    rel = item.get('relative_path') or item.get('path')
    expected = item.get('sha256')
    if not rel or not expected:
        continue
    candidate = (OP_BUNDLE / rel).resolve()
    op_checks[f'freeze_hash:{rel}'] = candidate.is_file() and sha256_file(candidate) == expected

# Verify primary hashes when fields are present.
for key, path in [
    ('model_sha256', OP_MODEL_PATH),
    ('calibrator_sha256', OP_CALIBRATOR_PATH),
    ('channel_schema_sha256', OP_SCHEMA_PATH),
    ('schema_sha256', OP_SCHEMA_PATH),
]:
    expected = op_policy.get(key) or op_freeze.get(key)
    if expected:
        op_checks[key] = sha256_file(path) == expected

failed = [k for k, v in op_checks.items() if not bool(v)]
atomic_json(AUDIT / 'observed_print_bundle_integrity.json', {
    'run': str(OP_RUN),
    'bundle': str(OP_BUNDLE),
    'replay': str(OP_REPLAY_PATH),
    'checks': op_checks,
})
require(not failed, f'59P observed-print bundle integrity failure: {failed}')

op_channels = op_schema.get('channels') or op_schema.get('ordered_channels')
require(isinstance(op_channels, list), '59P channel schema has no ordered channel list.')
require(op_channels == ['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge'],
        f'Unexpected 59P channel order: {op_channels}')
OP_PATCH = int(op_schema.get('patch', op_schema.get('patch_size', 33)))
require(OP_PATCH == 33, f'59P patch size is {OP_PATCH}, expected 33.')
OP_HALF = OP_PATCH // 2

OP_PRINT_THRESHOLD = float(op_policy.get('print_threshold', 0.995))
OP_ANNOTATION_THRESHOLD = float(op_policy.get('annotation_threshold', 0.995))
OP_PRINT_MARGIN = float(op_policy.get('print_margin', 0.990))
OP_MIN_PRINT_RUN = int(op_policy.get('minimum_print_run', op_policy.get('min_print_run', 2)))
OP_CONNECTIVITY = int(op_policy.get('connectivity', 8))

class TopologyCNN(nn.Module):
    def __init__(self, nc=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, 24, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(24), nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(48), nn.MaxPool2d(2),
            nn.Conv2d(48, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 2)
    def forward(self, x):
        return self.head(self.net(x).flatten(1))

op_model = TopologyCNN(nc=len(op_channels)).to(DEVICE)
state = torch.load(OP_MODEL_PATH, map_location=DEVICE)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
# Remove optional DataParallel prefix.
state = {k.replace('module.', '', 1): v for k, v in state.items()}
op_model.load_state_dict(state, strict=True)
op_model.eval()
op_calibrator = joblib.load(OP_CALIBRATOR_PATH)
require(hasattr(op_calibrator, 'predict'), '59P calibrator lacks predict().')

print('59P run:', OP_RUN)
print('59P replay:', OP_REPLAY_PATH)
print('59P integrity: PASS')

## 3. Resolve, verify, and load the 59R1 REMOVE bundle

In [ ]:
RM_BUNDLE_FILES = [
    'remove_model.joblib',
    'remove_calibrator.joblib',
    'predictor_schema.json',
    'policy.json',
    'reason_code_schema.json',
    'runtime_contract.json',
    'freeze_manifest.json',
]
RM_REPLAY_RELATIVE = Path('replay') / 'replay_test.json'

RM_RUN = discover_complete_run(
    REMOVE_ROOTS,
    REMOVE_RUN_OVERRIDE,
    RM_BUNDLE_FILES,
    RM_REPLAY_RELATIVE,
    '59R1 REMOVE deployment',
)
RM_BUNDLE = RM_RUN / 'bundle'
RM_REPLAY_PATH = RM_RUN / RM_REPLAY_RELATIVE

RM_MODEL_PATH = RM_BUNDLE / 'remove_model.joblib'
RM_CALIBRATOR_PATH = RM_BUNDLE / 'remove_calibrator.joblib'
RM_SCHEMA_PATH = RM_BUNDLE / 'predictor_schema.json'
RM_POLICY_PATH = RM_BUNDLE / 'policy.json'
RM_REASON_PATH = RM_BUNDLE / 'reason_code_schema.json'
RM_CONTRACT_PATH = RM_BUNDLE / 'runtime_contract.json'
RM_FREEZE_PATH = RM_BUNDLE / 'freeze_manifest.json'

rm_schema = read_json(RM_SCHEMA_PATH)
rm_policy = read_json(RM_POLICY_PATH)
rm_reasons = read_json(RM_REASON_PATH)
rm_contract = read_json(RM_CONTRACT_PATH)
rm_freeze = read_json(RM_FREEZE_PATH)
rm_replay = read_json(RM_REPLAY_PATH)

rm_checks = {
    'deployment_bundle_complete': rm_freeze.get('deployment_bundle_complete') is True,
    'replay_passed': rm_replay.get('passed') is True,
    'model_reload_exact': rm_replay.get('model_reload_exact') is True,
    'calibrator_reload_exact': rm_replay.get('calibrator_reload_exact') is True,
    'safe_protect_disabled': rm_policy.get('safe_protect_enabled', False) is False,
}
for item in rm_freeze.get('objects', []):
    rel = item.get('relative_path') or item.get('path')
    expected = item.get('sha256')
    if not rel or not expected:
        continue
    candidate = (RM_BUNDLE / rel).resolve()
    rm_checks[f'freeze_hash:{rel}'] = candidate.is_file() and sha256_file(candidate) == expected

for key, path in [
    ('model_sha256', RM_MODEL_PATH),
    ('calibrator_sha256', RM_CALIBRATOR_PATH),
    ('predictor_schema_sha256', RM_SCHEMA_PATH),
    ('schema_sha256', RM_SCHEMA_PATH),
    ('reason_code_schema_sha256', RM_REASON_PATH),
]:
    expected = rm_policy.get(key) or rm_freeze.get(key)
    if expected:
        rm_checks[key] = sha256_file(path) == expected

failed = [k for k, v in rm_checks.items() if not bool(v)]
atomic_json(AUDIT / 'remove_bundle_integrity.json', {
    'run': str(RM_RUN),
    'bundle': str(RM_BUNDLE),
    'replay': str(RM_REPLAY_PATH),
    'checks': rm_checks,
})
require(not failed, f'59R1 REMOVE bundle integrity failure: {failed}')

remove_predictors = rm_schema.get('ordered_predictors') or rm_schema.get('predictors')
require(isinstance(remove_predictors, list), '59R1 schema has no ordered predictor list.')
EXPECTED_REMOVE_PREDICTORS = [
    'observed_value','normalised_value','paper_value','mask_depth','boundary_distance',
    'distance_to_protected','edge','local_edge_fraction','local_dark_fraction',
    'local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction',
    'local_horizontal_fraction','local_vertical_fraction','local_compact_fraction',
    'p_print','component_id',
]
require(remove_predictors == EXPECTED_REMOVE_PREDICTORS,
        f'59R1 ordered predictor schema differs: {remove_predictors}')

remove_model = joblib.load(RM_MODEL_PATH)
remove_calibrator = joblib.load(RM_CALIBRATOR_PATH)
require(hasattr(remove_model, 'predict_proba'), '59R1 model lacks predict_proba().')
require(hasattr(remove_calibrator, 'predict'), '59R1 calibrator lacks predict().')
if hasattr(remove_model, 'n_features_in_'):
    require(int(remove_model.n_features_in_) == len(remove_predictors),
            '59R1 model feature count differs from schema.')

REMOVE_THRESHOLD = float(rm_policy.get('remove_threshold', rm_policy.get('threshold', 0.995)))
REMOVE_CONNECTIVITY = int(rm_policy.get('connectivity', 8))
REMOVE_MINIMUM_RUN = int(rm_policy.get('minimum_run', rm_policy.get('minimum_region_size', 2)))
REMOVE_REGION_MEAN_THRESHOLD = float(rm_policy.get('region_mean_threshold', REMOVE_THRESHOLD))

print('59R1 run:', RM_RUN)
print('59R1 replay:', RM_REPLAY_PATH)
print('59R1 predictors:', len(remove_predictors))
print('59R1 integrity: PASS')

## 4. Resolve the authoritative Notebook 07 reviewed cohort

In [ ]:
def first_existing_path(record, fields):
    for field in fields:
        value = record.get(field)
        if value is None or not str(value).strip():
            continue
        path = Path(str(value))
        if path.is_file():
            return path, field
    return None, None

review_records = []
parse_failures = []
for audit_path in sorted(REVIEW_AUDIT_DIR.glob('*.json')):
    try:
        record = read_json(audit_path)
        record['_audit_path'] = str(audit_path)
        review_records.append(record)
    except Exception as error:
        parse_failures.append({'review_audit_path': str(audit_path), 'error': str(error)})

pd.DataFrame(parse_failures).to_csv(AUDIT / 'review_audit_parse_failures.csv', index=False)
require(review_records, f'No readable Notebook 07 audit records found in {REVIEW_AUDIT_DIR}')
require(not parse_failures, f'{len(parse_failures)} Notebook 07 audits were unreadable.')

resolved = []
unresolved = []
for record in review_records:
    audit_path = Path(record['_audit_path'])
    observed_path, observed_field = first_existing_path(record, [
        'aligned_output','aligned_page','observed_output','aligned_image','observed_path'
    ])
    mask_path, mask_field = first_existing_path(record, [
        'corrected_mask_output','corrected_mask','reviewed_mask_output','reviewed_mask','mask_output'
    ])
    page = next((str(record.get(k)).strip() for k in ['page','page_id','page_name','canonical_page']
                 if record.get(k) is not None and str(record.get(k)).strip()), None)
    if page is None:
        page = observed_path.stem if observed_path else audit_path.stem
    if observed_path is None or mask_path is None:
        unresolved.append({'page': page, 'audit': str(audit_path), 'reason': 'missing observed or reviewed mask'})
        continue
    try:
        observed = read_gray(observed_path)
        seed = read_gray(mask_path)
        require(observed.shape == seed.shape, f'{page}: observed/mask shape mismatch.')
        require((seed > 0).any(), f'{page}: reviewed mask is empty.')
    except Exception as error:
        unresolved.append({'page': page, 'audit': str(audit_path), 'reason': str(error)})
        continue
    resolved.append({
        'page': page,
        'observed_path': str(observed_path),
        'reviewed_mask_path': str(mask_path),
        'review_audit_path': str(audit_path),
        'observed_sha256': sha256_file(observed_path),
        'reviewed_mask_sha256': sha256_file(mask_path),
        'height': observed.shape[0],
        'width': observed.shape[1],
        'reviewed_mask_pixels': int((seed > 0).sum()),
    })

pd.DataFrame(unresolved).to_csv(TABLES / 'unresolved_notebook07_reviews.csv', index=False)
roster = pd.DataFrame(resolved)
require(not roster.empty, 'No valid Notebook 07 reviewed pages resolved.')

# Collapse only byte-identical duplicates; conflicting duplicate page identities fail.
for page, group in roster.groupby('page'):
    if len(group) > 1:
        require(group.observed_sha256.nunique() == 1 and group.reviewed_mask_sha256.nunique() == 1,
                f'{page}: conflicting duplicate Notebook 07 records.')
roster = roster.sort_values(['page','review_audit_path']).drop_duplicates('page', keep='first').reset_index(drop=True)

if PAGE_ALLOWLIST is not None:
    requested = {str(x) for x in PAGE_ALLOWLIST}
    missing = sorted(requested - set(roster.page.astype(str)))
    require(not missing, f'Allowlisted pages unresolved: {missing}')
    roster = roster[roster.page.astype(str).isin(requested)].copy()

if REQUIRE_EXACT_151:
    require(len(roster) == EXPECTED_REVIEWED_PAGES,
            f'Expected {EXPECTED_REVIEWED_PAGES} reviewed pages; resolved {len(roster)}.')

roster.to_csv(TABLES / 'hitl_reviewed_roster.csv', index=False)
print('Reviewed pages:', len(roster))
display(roster[['page','observed_path','reviewed_mask_path','height','width']])

## 5. Canonical V7.8 page preparation

In [ ]:
def prepare_mask(seed, dilation=MASK_DILATION):
    mask = binary_mask(seed)
    if dilation > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*dilation+1, 2*dilation+1))
        mask = cv2.dilate(mask, kernel)
    return mask


def padded_components(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    h, w = mask.shape
    boxes = []
    for cid in range(1, n):
        x, y, bw, bh, area = stats[cid]
        if area >= 8:
            boxes.append((max(0, x-CROP_MARGIN), max(0, y-CROP_MARGIN),
                          min(w, x+bw+CROP_MARGIN), min(h, y+bh+CROP_MARGIN)))
    changed = True
    while changed:
        changed = False
        merged = []
        while boxes:
            ax0, ay0, ax1, ay1 = boxes.pop()
            remaining = []
            for bx0, by0, bx1, by1 in boxes:
                overlap = not (ax1 < bx0 or bx1 < ax0 or ay1 < by0 or by1 < ay0)
                if overlap:
                    ax0, ay0, ax1, ay1 = min(ax0,bx0), min(ay0,by0), max(ax1,bx1), max(ay1,by1)
                    changed = True
                else:
                    remaining.append((bx0,by0,bx1,by1))
            boxes = remaining
            merged.append((ax0,ay0,ax1,ay1))
        boxes = merged
    return boxes


def telea_reconstruct(observed, mask):
    result = observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im = observed[y0:y1, x0:x1]
        mk = mask[y0:y1, x0:x1]
        candidate = cv2.inpaint(im, mk, 3, cv2.INPAINT_TELEA)
        local = result[y0:y1, x0:x1]
        local[mk > 0] = candidate[mk > 0]
        result[y0:y1, x0:x1] = local
    return immutable(observed, result, mask)


def background_surface(observed, mask, sigma=45.0):
    visible = (mask == 0) & (observed >= 200)
    num = cv2.GaussianBlur(observed.astype(np.float32) * visible, (0,0), sigma)
    den = cv2.GaussianBlur(visible.astype(np.float32), (0,0), sigma)
    fallback = cv2.GaussianBlur(observed, (0,0), sigma)
    return np.clip(np.where(den > 0.02, num/np.maximum(den,1e-4), fallback), 0, 255).astype(np.uint8)


def support_maps(observed, mask):
    dark = (observed < 190) & (mask == 0)
    horizontal = cv2.morphologyEx(dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((1,35),np.uint8)) > 0
    vertical = cv2.morphologyEx(dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((15,1),np.uint8)) > 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark.astype(np.uint8), 8)
    compact = np.zeros_like(dark)
    for cid in range(1, n):
        x,y,w,h,area = stats[cid]
        if 3 <= area <= 650 and w <= 40 and h <= 40:
            compact[labels == cid] = True
    kernel = np.ones((2*BAND_WIDTH+1, 2*BAND_WIDTH+1), np.uint8)
    return (
        cv2.dilate(horizontal.astype(np.uint8), kernel) > 0,
        cv2.dilate(vertical.astype(np.uint8), kernel) > 0,
        cv2.dilate(compact.astype(np.uint8), kernel) > 0,
    )


def geometry_t80(observed, mask, telea):
    paper = background_surface(observed, mask)
    normalised = cv2.divide(observed, np.maximum(paper,1), scale=255)
    horizontal, vertical, compact = support_maps(observed, mask)
    inside = mask > 0
    preserve = inside & (horizontal | vertical | compact) & (observed < 200)
    remove = inside & ~preserve & (observed >= GEOMETRY_THRESHOLD) & (normalised >= GEOMETRY_THRESHOLD)
    ambiguous = inside & ~preserve & ~remove
    result = observed.copy()
    result[remove] = paper[remove]
    result[ambiguous] = telea[ambiguous]
    result[preserve] = observed[preserve]
    return immutable(observed, result, mask), {
        'paper': paper,
        'normalised': normalised,
        'horizontal': horizontal,
        'vertical': vertical,
        'compact': compact,
        'preserve': preserve,
        'remove': remove,
        'ambiguous': ambiguous,
    }

## 6. Generate calibrated page-shaped `p_print` from 59P

In [ ]:
def build_observed_print_channels(observed, prepared_mask, routes):
    return {
        'obs': observed.astype(np.float32) / 255.0,
        'norm': routes['normalised'].astype(np.float32) / 255.0,
        'mask': (prepared_mask > 0).astype(np.float32),
        'depth': cv2.distanceTransform((prepared_mask > 0).astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32),
        'horizontal': routes['horizontal'].astype(np.float32),
        'vertical': routes['vertical'].astype(np.float32),
        'compact': routes['compact'].astype(np.float32),
        'preserve': routes['preserve'].astype(np.float32),
        'remove': routes['remove'].astype(np.float32),
        'ambiguous': routes['ambiguous'].astype(np.float32),
        'edge': (cv2.Canny(observed, EDGE_LOW, EDGE_HIGH) > 0).astype(np.float32),
    }


def score_observed_print_page(page, observed, prepared_mask, routes):
    channels = build_observed_print_channels(observed, prepared_mask, routes)
    require(list(channels) == op_channels, f'{page}: observed-print channel order mismatch.')
    padded = {k: np.pad(v, OP_HALF, mode='reflect') for k, v in channels.items()}

    # Match the registered eligible population: dark, inside M3, not already preserved.
    eligible = (prepared_mask > 0) & (observed < DARK_THRESHOLD) & ~routes['preserve']
    ys, xs = np.where(eligible)

    raw_page = np.full(observed.shape, np.nan, dtype=np.float32)
    p_print = np.zeros(observed.shape, dtype=np.float32)
    provisional = np.full(observed.shape, 'NOT_ELIGIBLE', dtype=object)
    final = provisional.copy()

    if len(ys) == 0:
        return {
            'raw_page': raw_page, 'p_print': p_print, 'eligible': eligible,
            'provisional': provisional, 'final': final,
            'eligible_count': 0, 'singleton_demotions': 0,
        }

    raw = np.empty(len(ys), dtype=np.float64)
    op_model.eval()
    with torch.no_grad():
        for start in range(0, len(ys), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(ys))
            batch = np.stack([
                np.stack([padded[k][y:y+OP_PATCH, x:x+OP_PATCH] for k in op_channels])
                for y, x in zip(ys[start:stop], xs[start:stop])
            ]).astype(np.float32)
            logits = op_model(torch.from_numpy(batch).to(DEVICE, non_blocking=True))
            raw[start:stop] = torch.softmax(logits, dim=1)[:,1].cpu().numpy()

    require(np.isfinite(raw).all(), f'{page}: non-finite raw observed-print probabilities.')
    calibrated = np.asarray(op_calibrator.predict(raw), dtype=np.float64)
    require(calibrated.shape == raw.shape and np.isfinite(calibrated).all(),
            f'{page}: invalid calibrated p_print.')
    require(((calibrated >= 0) & (calibrated <= 1)).all(), f'{page}: p_print outside [0,1].')

    raw_page[ys, xs] = raw.astype(np.float32)
    p_print[ys, xs] = calibrated.astype(np.float32)

    p_annotation = 1.0 - calibrated
    decisions = np.full(len(calibrated), 'UNCERTAIN', dtype=object)
    decisions[p_annotation >= OP_ANNOTATION_THRESHOLD] = 'ANNOTATION'
    decisions[(calibrated >= OP_PRINT_THRESHOLD) & ((calibrated - p_annotation) >= OP_PRINT_MARGIN)] = 'PRINT'
    provisional[ys, xs] = decisions
    final[ys, xs] = decisions

    # Page-global no-singleton / minimum connected-run rule.
    print_map = np.zeros(observed.shape, np.uint8)
    print_hits = decisions == 'PRINT'
    print_map[ys[print_hits], xs[print_hits]] = 1
    n, labels, stats, _ = cv2.connectedComponentsWithStats(print_map, OP_CONNECTIVITY)
    singleton_demotions = 0
    for cid in range(1, n):
        if int(stats[cid, cv2.CC_STAT_AREA]) < OP_MIN_PRINT_RUN:
            region = labels == cid
            final[region] = 'UNCERTAIN'
            singleton_demotions += int(region.sum())

    return {
        'raw_page': raw_page, 'p_print': p_print, 'eligible': eligible,
        'provisional': provisional, 'final': final,
        'eligible_count': len(ys), 'singleton_demotions': singleton_demotions,
    }

## 7. Exact 17-field 59R1 predictor builder

In [ ]:
def local_fraction(mask, radius=LOCAL_RADIUS):
    size = 2 * int(radius) + 1
    return cv2.boxFilter(
        np.asarray(mask, dtype=np.float32),
        ddepth=cv2.CV_32F,
        ksize=(size, size),
        normalize=True,
        borderType=cv2.BORDER_REFLECT,
    ).astype(np.float32)


def distance_to_true(mask):
    target = np.asarray(mask, dtype=bool)
    if target.any():
        return cv2.distanceTransform((~target).astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    h, w = target.shape
    return np.full(target.shape, np.hypot(h, w), dtype=np.float32)


def mask_boundary(mask):
    inside = np.asarray(mask, dtype=bool)
    eroded = cv2.erode(
        inside.astype(np.uint8), np.ones((3,3),np.uint8), iterations=1,
        borderType=cv2.BORDER_CONSTANT, borderValue=0
    ).astype(bool)
    return inside & ~eroded


def exact_59r1_predictor_frame(page, observed, prepared_mask, routes, p_print):
    inside = prepared_mask > 0
    require(inside.any(), f'{page}: M3 is empty.')
    require(p_print.shape == observed.shape, f'{page}: p_print shape mismatch.')

    mask_depth = cv2.distanceTransform(inside.astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    boundary = mask_boundary(inside)
    boundary_distance = distance_to_true(boundary)
    distance_to_protected = distance_to_true(routes['preserve'])
    edge = cv2.Canny(observed, EDGE_LOW, EDGE_HIGH) > 0
    _, component_ids = cv2.connectedComponents(inside.astype(np.uint8), connectivity=8)

    arrays = {
        'observed_value': observed.astype(np.float32),
        'normalised_value': routes['normalised'].astype(np.float32),
        'paper_value': routes['paper'].astype(np.float32),
        'mask_depth': mask_depth,
        'boundary_distance': boundary_distance,
        'distance_to_protected': distance_to_protected,
        'edge': edge.astype(np.float32),
        'local_edge_fraction': local_fraction(edge),
        'local_dark_fraction': local_fraction(observed < 190),
        'local_preserve_fraction': local_fraction(routes['preserve']),
        'local_remove_fraction': local_fraction(routes['remove']),
        'local_ambiguous_fraction': local_fraction(routes['ambiguous']),
        'local_horizontal_fraction': local_fraction(routes['horizontal']),
        'local_vertical_fraction': local_fraction(routes['vertical']),
        'local_compact_fraction': local_fraction(routes['compact']),
        'p_print': p_print.astype(np.float32),
        'component_id': component_ids.astype(np.float32),
    }

    require(list(arrays) == remove_predictors,
            f'{page}: runtime predictor names/order differ from frozen 59R1 schema.')
    for name, array in arrays.items():
        require(array.shape == observed.shape, f'{page}: predictor {name} shape mismatch.')

    y, x = np.where(inside)
    frame = pd.DataFrame({'page': page, 'y': y.astype(np.int32), 'x': x.astype(np.int32)})
    for name in remove_predictors:
        frame[name] = arrays[name][y, x]

    X = frame.loc[:, remove_predictors].copy()
    for name in remove_predictors:
        X[name] = pd.to_numeric(X[name], errors='coerce')
    values = X.to_numpy(dtype=np.float64)
    require(not np.isinf(values).any(), f'{page}: infinite 59R1 predictor value.')

    return {
        'frame': frame,
        'X': X,
        'inside': inside,
        'y': y,
        'x': x,
        'component_ids': component_ids,
        'boundary': boundary,
        'edge': edge,
        'nan_counts': {c:int(X[c].isna().sum()) for c in X if X[c].isna().any()},
    }


def score_remove_page(page, built):
    raw = np.asarray(remove_model.predict_proba(built['X'])[:,1], dtype=np.float64)
    require(raw.shape == (len(built['frame']),) and np.isfinite(raw).all(),
            f'{page}: invalid raw REMOVE probabilities.')
    p_remove = np.asarray(remove_calibrator.predict(raw), dtype=np.float64)
    require(p_remove.shape == raw.shape and np.isfinite(p_remove).all(),
            f'{page}: invalid calibrated REMOVE probabilities.')
    require(((p_remove >= 0) & (p_remove <= 1)).all(), f'{page}: p_remove outside [0,1].')

    raw_page = np.full(built['inside'].shape, np.nan, dtype=np.float32)
    p_remove_page = np.full(built['inside'].shape, np.nan, dtype=np.float32)
    raw_page[built['y'], built['x']] = raw.astype(np.float32)
    p_remove_page[built['y'], built['x']] = p_remove.astype(np.float32)

    provisional = built['inside'] & np.isfinite(p_remove_page) & (p_remove_page >= REMOVE_THRESHOLD)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(provisional.astype(np.uint8), REMOVE_CONNECTIVITY)
    safe_remove = np.zeros(built['inside'].shape, dtype=bool)
    regions = []
    for cid in range(1, n):
        region = labels == cid
        size = int(stats[cid, cv2.CC_STAT_AREA])
        probs = p_remove_page[region]
        mean = float(probs.mean())
        minimum = float(probs.min())
        passes = bool(
            size >= REMOVE_MINIMUM_RUN
            and mean >= REMOVE_REGION_MEAN_THRESHOLD
            and minimum >= REMOVE_THRESHOLD
        )
        if passes:
            safe_remove[region] = True
        regions.append({
            'page': page, 'region_id': cid, 'pixels': size,
            'mean_p_remove': mean, 'minimum_p_remove': minimum,
            'passes_safe_remove': passes,
        })

    require(not np.any(safe_remove & ~built['inside']), f'{page}: SAFE_REMOVE escaped M3.')
    return {
        'raw': raw,
        'p_remove': p_remove,
        'raw_page': raw_page,
        'p_remove_page': p_remove_page,
        'provisional': provisional,
        'safe_remove': safe_remove,
        'regions': regions,
    }

## 8. Process all reviewed pages

In [ ]:
page_rows = []
region_rows = []
pixel_tables = []
failures = []

for row in roster.itertuples(index=False):
    page = str(row.page)
    print('Processing', page)
    page_dir = PAGES_OUT / page
    page_dir.mkdir(parents=True, exist_ok=False)
    try:
        observed = read_gray(Path(row.observed_path))
        reviewed_seed = read_gray(Path(row.reviewed_mask_path))
        require(observed.shape == reviewed_seed.shape, f'{page}: observed/reviewed mask mismatch.')

        m3_u8 = prepare_mask(reviewed_seed, MASK_DILATION)
        telea_m3 = telea_reconstruct(observed, m3_u8)
        v78_m3, routes = geometry_t80(observed, m3_u8, telea_m3)

        op = score_observed_print_page(page, observed, m3_u8, routes)
        built = exact_59r1_predictor_frame(page, observed, m3_u8, routes, op['p_print'])
        rm = score_remove_page(page, built)

        m3 = built['inside']
        safe_remove = rm['safe_remove']
        m5_remove = m3 & ~safe_remove
        require(np.array_equal(m5_remove, m3 & ~safe_remove), f'{page}: M5_REMOVE identity failed.')

        m5_u8 = (m5_remove.astype(np.uint8) * 255)
        telea_m5 = telea_reconstruct(observed, m5_u8)
        corrected_m5, m5_routes = geometry_t80(observed, m5_u8, telea_m5)
        require(np.array_equal(corrected_m5[~m3], observed[~m3]), f'{page}: output changed outside M3.')

        # Save images.
        cv2.imwrite(str(page_dir/'observed.png'), observed)
        cv2.imwrite(str(page_dir/'reviewed_seed_mask.png'), binary_mask(reviewed_seed))
        cv2.imwrite(str(page_dir/'m3_prepared_mask.png'), m3_u8)
        cv2.imwrite(str(page_dir/'v7_8_m3.png'), v78_m3)
        cv2.imwrite(str(page_dir/'safe_remove.png'), safe_remove.astype(np.uint8)*255)
        cv2.imwrite(str(page_dir/'m5_remove_mask.png'), m5_u8)
        cv2.imwrite(str(page_dir/'corrected_m5_remove.png'), corrected_m5)

        # Save exact arrays without relying on image quantisation.
        np.savez_compressed(
            page_dir/'runtime_arrays.npz',
            p_print=op['p_print'].astype(np.float32),
            raw_print_probability=op['raw_page'].astype(np.float32),
            observed_print_eligible=op['eligible'].astype(np.uint8),
            raw_remove_probability=rm['raw_page'].astype(np.float32),
            p_remove=rm['p_remove_page'].astype(np.float32),
            provisional_remove=rm['provisional'].astype(np.uint8),
            safe_remove=safe_remove.astype(np.uint8),
            m3=m3.astype(np.uint8),
            m5_remove=m5_remove.astype(np.uint8),
        )

        pixel = built['frame'].copy()
        pixel['raw_remove_probability'] = rm['raw']
        pixel['p_remove'] = rm['p_remove']
        pixel['safe_remove'] = safe_remove[built['y'], built['x']]
        pixel['p_print_generated_by_59p'] = op['p_print'][built['y'], built['x']]
        pixel_tables.append(pixel)
        region_rows.extend(rm['regions'])

        page_record = {
            'page': page,
            'observed_path': row.observed_path,
            'reviewed_mask_path': row.reviewed_mask_path,
            'height': observed.shape[0],
            'width': observed.shape[1],
            'reviewed_seed_pixels': int((reviewed_seed > 0).sum()),
            'm3_pixels': int(m3.sum()),
            'observed_print_eligible_pixels': int(op['eligible_count']),
            'observed_print_final_print_pixels': int((op['final'] == 'PRINT').sum()),
            'observed_print_singleton_demotions': int(op['singleton_demotions']),
            'safe_remove_pixels': int(safe_remove.sum()),
            'm5_remove_pixels': int(m5_remove.sum()),
            'outside_m3_drift': int(np.sum(corrected_m5[~m3] != observed[~m3])),
            'observed_sha256': sha256_file(Path(row.observed_path)),
            'reviewed_mask_sha256': sha256_file(Path(row.reviewed_mask_path)),
            'p_print_array_sha256': array_sha256(op['p_print']),
            'p_remove_array_sha256': array_sha256(rm['p_remove_page']),
            'safe_remove_array_sha256': array_sha256(safe_remove.astype(np.uint8)),
            'm5_remove_array_sha256': array_sha256(m5_remove.astype(np.uint8)),
            'corrected_output_sha256': sha256_file(page_dir/'corrected_m5_remove.png'),
            'predictor_nan_counts': built['nan_counts'],
        }
        atomic_json(page_dir/'page_manifest.json', page_record)
        page_rows.append(page_record)

    except Exception as error:
        failure = {'page': page, 'error': repr(error)}
        atomic_json(page_dir/'failure.json', failure)
        failures.append(failure)

failure_df = pd.DataFrame(failures)
failure_df.to_csv(TABLES/'page_failures.csv', index=False)
require(not failures, f'{len(failures)} page(s) failed. See page_failures.csv.')

page_df = pd.DataFrame(page_rows)
page_df.to_csv(TABLES/'page_summary.csv', index=False)
pd.DataFrame(region_rows).to_csv(TABLES/'safe_remove_regions.csv', index=False)
all_pixels = pd.concat(pixel_tables, ignore_index=True)
all_pixels.to_parquet(TABLES/'pixel_predictions.parquet', index=False)

print('Processed pages:', len(page_df))
print('SAFE_REMOVE pixels:', int(page_df.safe_remove_pixels.sum()))
print('Outside-M3 drift:', int(page_df.outside_m3_drift.sum()))
display(page_df[['page','m3_pixels','safe_remove_pixels','m5_remove_pixels','outside_m3_drift']])

## 9. Combined runtime identity, integrity verdict, and final upload package

In [ ]:
require(len(page_df) == len(roster), 'Processed page count differs from reviewed roster.')
require(int(page_df.outside_m3_drift.sum()) == 0, 'Outside-M3 drift was detected.')

combined_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook': '60R2',
    'supervised_only': True,
    'production_authority': False,
    'notebook07_reviewed_mask_is_intervention_authority': True,
    'notebook06_yolo_confidence_used_as_p_print': False,
    'observed_print': {
        'run': str(OP_RUN),
        'model_sha256': sha256_file(OP_MODEL_PATH),
        'calibrator_sha256': sha256_file(OP_CALIBRATOR_PATH),
        'schema_sha256': sha256_file(OP_SCHEMA_PATH),
        'policy_sha256': sha256_file(OP_POLICY_PATH),
        'replay_sha256': sha256_file(OP_REPLAY_PATH),
        'policy_id': op_policy.get('policy_id'),
        'compatibility_id': op_policy.get('compatibility_id') or op_freeze.get('compatibility_id'),
    },
    'remove': {
        'run': str(RM_RUN),
        'model_sha256': sha256_file(RM_MODEL_PATH),
        'calibrator_sha256': sha256_file(RM_CALIBRATOR_PATH),
        'schema_sha256': sha256_file(RM_SCHEMA_PATH),
        'policy_sha256': sha256_file(RM_POLICY_PATH),
        'replay_sha256': sha256_file(RM_REPLAY_PATH),
        'policy_id': rm_policy.get('policy_id'),
        'compatibility_id': rm_policy.get('compatibility_id') or rm_freeze.get('compatibility_id'),
    },
    'constants': {
        'mask_dilation': MASK_DILATION,
        'geometry_threshold': GEOMETRY_THRESHOLD,
        'band_width': BAND_WIDTH,
        'dark_threshold': DARK_THRESHOLD,
        'local_radius': LOCAL_RADIUS,
        'remove_threshold': REMOVE_THRESHOLD,
        'remove_connectivity': REMOVE_CONNECTIVITY,
        'remove_minimum_run': REMOVE_MINIMUM_RUN,
        'remove_region_mean_threshold': REMOVE_REGION_MEAN_THRESHOLD,
    },
    'pages': len(page_df),
    'm3_pixels': int(page_df.m3_pixels.sum()),
    'safe_remove_pixels': int(page_df.safe_remove_pixels.sum()),
    'm5_remove_pixels': int(page_df.m5_remove_pixels.sum()),
    'outside_m3_drift': int(page_df.outside_m3_drift.sum()),
}
combined_contract['combined_runtime_id'] = object_hash(combined_contract)
atomic_json(OUT/'runtime_manifest.json', combined_contract)

software = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'device': DEVICE,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'joblib': joblib.__version__,
}
atomic_json(OUT/'software_environment.json', software)

verdict = {
    'verdict': 'PASS',
    'reviewed_pages': len(roster),
    'processed_pages': len(page_df),
    'page_failures': len(failures),
    'outside_m3_drift': int(page_df.outside_m3_drift.sum()),
    'observed_print_bundle_integrity': True,
    'remove_bundle_integrity': True,
    'exact_17_predictor_schema': list(remove_predictors) == EXPECTED_REMOVE_PREDICTORS,
    'p_print_generated_in_memory': True,
    'pre_generated_p_print_dependency': False,
    'safe_protect_enabled': False,
    'production_authority': False,
    'combined_runtime_id': combined_contract['combined_runtime_id'],
}
atomic_json(OUT/'verdict.json', verdict)

summary_lines = [
    'NOTEBOOK 60R2 VERIFIED INTEGRATED RUNTIME',
    '',
    f'Run: {OUT}',
    f'Reviewed pages: {len(roster)}',
    f'Processed pages: {len(page_df)}',
    f'M3 pixels: {int(page_df.m3_pixels.sum())}',
    f'SAFE_REMOVE pixels: {int(page_df.safe_remove_pixels.sum())}',
    f'M5_REMOVE pixels: {int(page_df.m5_remove_pixels.sum())}',
    f'Outside-M3 drift: {int(page_df.outside_m3_drift.sum())}',
    f'Combined runtime ID: {combined_contract["combined_runtime_id"]}',
    '',
    'p_print is generated directly from the verified 59P bundle.',
    'All 17 frozen 59R1 predictors are constructed in the saved order.',
    'Notebook 07 reviewed masks are the sole intervention authority.',
    'SAFE_PROTECT is disabled. Production authority is false.',
]
(OUT/'executive_summary.txt').write_text('\n'.join(summary_lines))

# Curated upload package excludes durable binary model objects.
curated = [
    OUT/'executive_summary.txt',
    OUT/'runtime_manifest.json',
    OUT/'verdict.json',
    OUT/'software_environment.json',
    AUDIT/'observed_print_bundle_integrity.json',
    AUDIT/'remove_bundle_integrity.json',
    TABLES/'hitl_reviewed_roster.csv',
    TABLES/'page_summary.csv',
    TABLES/'safe_remove_regions.csv',
    TABLES/'page_failures.csv',
]

manifest_rows = []
for src in curated:
    require(src.exists(), f'Missing curated upload artefact: {src}')
    rel = src.relative_to(OUT)
    dst = UPLOAD / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    manifest_rows.append({
        'relative_path': str(rel),
        'bytes': src.stat().st_size,
        'sha256': sha256_file(src),
    })

upload_manifest = pd.DataFrame(manifest_rows)
upload_manifest.to_csv(UPLOAD/'upload_manifest.csv', index=False)
(UPLOAD/'README.txt').write_text(
    'Notebook 60R2 curated review package.\n'
    'The 59P .pt model and both .joblib objects remain in their durable run directories and are not copied here.\n'
    'Use runtime_manifest.json for their verified source paths, hashes, policy IDs and compatibility IDs.\n'
)

zip_path = OUT/'notebook60r2_upload_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(UPLOAD))

print('\n'.join(summary_lines))
print('Upload ZIP:', zip_path)